In [4]:
from helper import titanic_conn, q, show_tables, show_columns

conn = titanic_conn()
print(show_tables(conn))
show_columns(conn, 'titanic')

['titanic']


,name,type
0,PassengerId,INTEGER
1,Survived,INTEGER
2,Pclass,INTEGER
3,Name,TEXT
4,Sex,TEXT
5,Age,REAL
6,SibSp,INTEGER
7,Parch,INTEGER
8,Ticket,TEXT
9,Fare,REAL


In [14]:
sql = '''
select count(*) as n from titanic
'''
sql = '''
SELECT PassengerId, Name, Sex, Age
FROM titanic
LIMIT 3
'''
sql = '''
SELECT Name, Pclass FROM titanic 
-- SELECT count(*) FROM titanic 
WHERE Sex = 'female' AND Pclass = 1
LIMIT 5
'''
# 문제1: Sex가 female 이면서 Pclass 가 1 인것 -> 이름, Sex, Pclass 출력
# 문제2: 문제1이 몇명인지 -> count(*)
result = q(conn, sql)
result

,Name,Pclass
0,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1
1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1
2,"Bonnell, Miss. Elizabeth",1
3,"Spencer, Mrs. William Augustus (Marie Eugenie)",1
4,"Harper, Mrs. Henry Sleeper (Myna Haxtun)",1


In [16]:
sql = '''
SELECT Name, Fare FROM titanic 
ORDER BY FARE DESC  -- DESC 를 빼거나, ASC
LIMIT 5
'''
result = q(conn, sql)
result

,Name,Fare
0,"Ward, Miss. Anna",512.3292
1,"Cardeza, Mr. Thomas Drake Martinez",512.3292
2,"Lesurer, Mr. Gustave J",512.3292
3,"Fortune, Mr. Charles Alexander",263.0000
4,"Fortune, Miss. Mabel Helen",263.0000


## DISTINCT

In [29]:
# DISTINCT
sql = '''
SELECT DISTINCT Embarked FROM titanic
'''
# LIKE
sql = '''
SELECT COUNT(*) AS n FROM titanic
WHERE Name LIKE '%Miss%'
'''
# IN, BETWEEN
sql = '''
SELECT COUNT(*) AS n FROM titanic 
WHERE Embarked IN ('C', 'Q') AND Age BETWEEN 20 AND 30
'''
# Null
sql = '''
SELECT COUNT(*) AS n FROM titanic 
WHERE Age IS NULL  -- = 요거 사용했을때와 비교   | Age IS NOT NULL
'''
# Case 
sql = '''
SELECT Name, Age, 
    CASE WHEN Age < 12 THEN 'child'
    ELSE 'adult'
    END as age_group
FROM titanic
limit 3
'''
# 10시 45분까지 휴식
result = q(conn, sql)
result

,Name,Age,age_group
0,"Braund, Mr. Owen Harris",22.0,adult
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",38.0,adult
2,"Heikkinen, Miss. Laina",26.0,adult


In [ ]:
# 문제1 : 가장 나이 많은 승객의 이름과 나이
# 문제2 : 3등석 여성 중에 생존자 수 
# 문제3 : 나이를 child(<12), adult, unknown(NULL) 3그룹으로 만들기

In [34]:
sql = '''
SELECT Name, Age FROM titanic
ORDER BY Age DESC LIMIT 1
'''
sql = '''
SELECT COUNT(*) FROM titanic
WHERE Survived = 1 AND Sex = 'female' AND Pclass = 3
'''
sql = '''
SELECT CASE WHEN Age < 12 THEN 'child'
            WHEN Age IS NULL THEN 'unknonw'
            ELSE 'adult' END AS age_group
FROM titanic 
'''
result = q(conn, sql)


# Group BY

In [38]:
sql = '''
SELECT COUNT(*) as n,
    COUNT(age) as n_age,
    ROUND(AVG(Age), 1) as avg_age,
    MIN(Fare) as min_fare,
    MAX(Fare) as max_fare,
    SUM(Survived) as n_survived
FROM titanic
'''
q(conn, sql)


,n,n_age,avg_age,min_fare,max_fare,n_survived
0,891,714,29.7,0.0,512.3292,342


In [40]:
sql = '''
SELECT Sex, 
    Pclass,
    COUNT(*) as n,
    ROUND(AVG(Survived), 2) as survival_rate
FROM titanic
GROUP BY Sex, Pclass
'''
q(conn, sql)

,Sex,Pclass,n,survival_rate
0,female,1,94,0.97
1,female,2,76,0.92
2,female,3,144,0.50
3,male,1,122,0.37
4,male,2,108,0.16
5,male,3,347,0.14


In [43]:
sql = '''
SELECT Embarked, COUNT(*) as n
FROM titanic
GROUP BY Embarked
HAVING COUNT(*) >= 100
'''
q(conn, sql)

,Embarked,n
0,C,168
1,S,644


## 6. 실행 순서 — 쓰는 순서와 다르다 (중요!)

```
쓰는 순서 : SELECT → FROM → WHERE → GROUP BY → HAVING → ORDER BY → LIMIT
실행 순서 : FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT
```

1. `FROM` 표를 가져와서
2. `WHERE` 로 행을 거르고
3. `GROUP BY` 로 묶고
4. `HAVING` 으로 그룹을 거르고
5. `SELECT` 로 보여줄 열을 계산하고
6. `ORDER BY` 로 정렬하고
7. `LIMIT` 으로 자른다

그래서 `WHERE` 안에서는 `AVG()` 같은 집계를 쓸 수 없고(아직 그룹이 없음), `SELECT` 에서 붙인 별칭(`AS avg_age`)을 `ORDER BY` 에서는 쓸 수 있습니다.

# JOIN